# Road extraction — train the 4 models (identical recipe)

**Setup (once):**
1. *Settings -> Accelerator -> GPU T4 x2* (NOT P100 — Kaggle's current PyTorch
   no longer supports its sm_60 architecture).
2. *Settings -> Internet -> ON* (pip + ImageNet encoder weights).
3. *Add Input -> Your Work* -> the prepare-data notebook's output (the `tiles/` folder).
4. Set `MODELS` below. The two models train in parallel, one per T4 (~4h wall);
   run the notebook twice:
   session A `['unet', 'unetpp']`, session B `['deeplabv3plus', 'linknet']`.
5. *Save Version -> Save & Run All*. Download `runs/` from the output when done.


In [ ]:
%pip install -q segmentation-models-pytorch albumentations


In [ ]:
import os; os.makedirs('roadx/data', exist_ok=True); os.makedirs('configs', exist_ok=True); open('roadx/__init__.py','w').close(); open('roadx/data/__init__.py','w').close()


In [ ]:
%%writefile roadx/data/dataset.py
from pathlib import Path

import albumentations as A
import numpy as np
import torch
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import Dataset

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def train_transform() -> A.Compose:
    return A.Compose(
        [
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]
    )


def eval_transform() -> A.Compose:
    return A.Compose(
        [
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]
    )


class RoadDataset(Dataset):
    """512x512 image tiles with binary road masks."""

    def __init__(self, root: Path, split: str, train: bool, limit: int | None = None):
        self.img_dir = Path(root) / split / "images"
        self.msk_dir = Path(root) / split / "masks"
        self.items = sorted(p.name for p in self.img_dir.glob("*.png"))
        if limit:
            self.items = self.items[:limit]
        if not self.items:
            raise FileNotFoundError(f"no tiles found in {self.img_dir}")
        self.tf = train_transform() if train else eval_transform()

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        name = self.items[idx]
        img = np.asarray(Image.open(self.img_dir / name).convert("RGB"))
        msk = (np.asarray(Image.open(self.msk_dir / name).convert("L")) > 127).astype(np.float32)
        out = self.tf(image=img, mask=msk)
        return out["image"], out["mask"].unsqueeze(0)


In [ ]:
%%writefile roadx/models.py
import segmentation_models_pytorch as smp
import torch

ARCHS = {
    "unet": smp.Unet,
    "unetpp": smp.UnetPlusPlus,
    "deeplabv3plus": smp.DeepLabV3Plus,
    "linknet": smp.Linknet,
}


def build_model(
    name: str, encoder: str = "resnet34", encoder_weights: str | None = "imagenet"
) -> torch.nn.Module:
    if name not in ARCHS:
        raise ValueError(f"unknown model '{name}', choose from {sorted(ARCHS)}")
    return ARCHS[name](
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=1,
    )


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


In [ ]:
%%writefile roadx/losses.py
import segmentation_models_pytorch as smp
import torch
from torch import nn


class DiceBCELoss(nn.Module):
    """Dice + BCE, the standard combination for thin-structure segmentation."""

    def __init__(self, dice_weight: float = 0.5):
        super().__init__()
        self.dice = smp.losses.DiceLoss(mode="binary")
        self.bce = nn.BCEWithLogitsLoss()
        self.w = dice_weight

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        return self.w * self.dice(logits, target) + (1 - self.w) * self.bce(logits, target)


In [ ]:
%%writefile roadx/metrics.py
import torch

EPS = 1e-7


class SegMetrics:
    """Accumulates confusion counts over batches, reports IoU/F1/precision/recall."""

    def __init__(self, threshold: float = 0.5):
        self.threshold = threshold
        self.tp = self.fp = self.fn = self.tn = 0.0

    @torch.no_grad()
    def update(self, logits: torch.Tensor, target: torch.Tensor) -> None:
        pred = (torch.sigmoid(logits) > self.threshold).float()
        t = target.float()
        self.tp += (pred * t).sum().item()
        self.fp += (pred * (1 - t)).sum().item()
        self.fn += ((1 - pred) * t).sum().item()
        self.tn += ((1 - pred) * (1 - t)).sum().item()

    def compute(self) -> dict[str, float]:
        precision = self.tp / (self.tp + self.fp + EPS)
        recall = self.tp / (self.tp + self.fn + EPS)
        return {
            "iou": self.tp / (self.tp + self.fp + self.fn + EPS),
            "f1": 2 * precision * recall / (precision + recall + EPS),
            "precision": precision,
            "recall": recall,
        }


In [ ]:
%%writefile roadx/train.py
"""Train one segmentation model. The architecture is a config/flag choice so all
four models train under identical conditions.

    python -m roadx.train --model unet
    python -m roadx.train --model deeplabv3plus --epochs 5 --limit 200   # sanity run
"""

import argparse
import csv
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
import yaml
from torch.utils.data import DataLoader
from tqdm import tqdm

from roadx.data.dataset import RoadDataset
from roadx.losses import DiceBCELoss
from roadx.metrics import SegMetrics
from roadx.models import build_model, pick_device


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def run_epoch(model, loader, loss_fn, device, optimizer=None) -> dict[str, float]:
    training = optimizer is not None
    model.train() if training else model.eval()
    metrics = SegMetrics()
    total_loss = 0.0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for img, msk in tqdm(loader, leave=False):
            img, msk = img.to(device), msk.to(device)
            logits = model(img)
            loss = loss_fn(logits, msk)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * img.size(0)
            metrics.update(logits, msk)
    out = metrics.compute()
    out["loss"] = total_loss / len(loader.dataset)
    return out


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--model", required=True, help="unet | unetpp | deeplabv3plus | linknet")
    p.add_argument("--config", type=Path, default=None, help="defaults to configs/<model>.yaml")
    p.add_argument("--data-dir", type=Path, default=None)
    p.add_argument("--epochs", type=int, default=None)
    p.add_argument("--batch-size", type=int, default=None)
    p.add_argument("--limit", type=int, default=None, help="cap train samples (sanity runs)")
    p.add_argument("--out", type=Path, default=Path("runs"))
    args = p.parse_args()

    cfg_path = args.config or Path("configs") / f"{args.model}.yaml"
    cfg = yaml.safe_load(cfg_path.read_text())
    if args.data_dir:
        cfg["data_dir"] = str(args.data_dir)
    if args.epochs:
        cfg["epochs"] = args.epochs
    if args.batch_size:
        cfg["batch_size"] = args.batch_size

    seed_everything(cfg.get("seed", 42))
    device = pick_device()
    print(f"model={cfg['model']} encoder={cfg['encoder']} device={device.type}")

    train_ds = RoadDataset(cfg["data_dir"], "train", train=True, limit=args.limit)
    valid_ds = RoadDataset(cfg["data_dir"], "valid", train=False)
    print(f"train tiles={len(train_ds)} valid tiles={len(valid_ds)}")

    pin = device.type == "cuda"
    train_dl = DataLoader(
        train_ds, cfg["batch_size"], shuffle=True,
        num_workers=cfg.get("num_workers", 4), pin_memory=pin, drop_last=True,
    )
    valid_dl = DataLoader(
        valid_ds, cfg["batch_size"], shuffle=False,
        num_workers=cfg.get("num_workers", 4), pin_memory=pin,
    )

    model = build_model(cfg["model"], cfg["encoder"], cfg.get("encoder_weights", "imagenet"))
    model.to(device)
    loss_fn = DiceBCELoss()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=float(cfg["lr"]), weight_decay=float(cfg.get("weight_decay", 1e-4))
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"])

    run_dir = args.out / cfg["model"]
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "config.json").write_text(json.dumps(cfg, indent=2))
    log_path = run_dir / "log.csv"
    best_iou = -1.0

    with open(log_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "lr", "train_loss", "train_iou", "val_loss", "val_iou", "val_f1", "seconds"])
        for epoch in range(1, cfg["epochs"] + 1):
            t0 = time.time()
            tr = run_epoch(model, train_dl, loss_fn, device, optimizer)
            va = run_epoch(model, valid_dl, loss_fn, device)
            scheduler.step()
            dt = time.time() - t0
            writer.writerow([
                epoch, f"{optimizer.param_groups[0]['lr']:.2e}",
                f"{tr['loss']:.4f}", f"{tr['iou']:.4f}",
                f"{va['loss']:.4f}", f"{va['iou']:.4f}", f"{va['f1']:.4f}", f"{dt:.1f}",
            ])
            f.flush()
            marker = ""
            if va["iou"] > best_iou:
                best_iou = va["iou"]
                torch.save(
                    {"model": cfg["model"], "encoder": cfg["encoder"],
                     "state_dict": model.state_dict(), "val_iou": best_iou, "epoch": epoch},
                    run_dir / "best.pt",
                )
                marker = " *best*"
            print(
                f"epoch {epoch:3d}/{cfg['epochs']} "
                f"train loss {tr['loss']:.4f} iou {tr['iou']:.4f} | "
                f"val loss {va['loss']:.4f} iou {va['iou']:.4f} f1 {va['f1']:.4f} "
                f"({dt:.0f}s){marker}"
            )

    print(f"done. best val IoU {best_iou:.4f} -> {run_dir / 'best.pt'}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile configs/unet.yaml
model: unet
encoder: resnet34
encoder_weights: imagenet
img_size: 512
batch_size: 8
epochs: 30
lr: 3.0e-4
weight_decay: 1.0e-4
num_workers: 4
seed: 42
data_dir: data/tiles


In [ ]:
%%writefile configs/unetpp.yaml
model: unetpp
encoder: resnet34
encoder_weights: imagenet
img_size: 512
batch_size: 8
epochs: 30
lr: 3.0e-4
weight_decay: 1.0e-4
num_workers: 4
seed: 42
data_dir: data/tiles


In [ ]:
%%writefile configs/deeplabv3plus.yaml
model: deeplabv3plus
encoder: resnet34
encoder_weights: imagenet
img_size: 512
batch_size: 8
epochs: 30
lr: 3.0e-4
weight_decay: 1.0e-4
num_workers: 4
seed: 42
data_dir: data/tiles


In [ ]:
%%writefile configs/linknet.yaml
model: linknet
encoder: resnet34
encoder_weights: imagenet
img_size: 512
batch_size: 8
epochs: 30
lr: 3.0e-4
weight_decay: 1.0e-4
num_workers: 4
seed: 42
data_dir: data/tiles


In [ ]:
import os

MODELS = ['deeplabv3plus', 'linknet']  # session B
BATCH_SIZE = 8  # identical for all models; 16 OOMs unetpp on a 16 GB T4

# Kaggle mounts inputs at nested paths (/kaggle/input/datasets/<owner>/<slug>,
# notebook outputs similarly), so search recursively for the tiles folder.
hits = [d for d, _, _ in os.walk('/kaggle/input') if os.path.isdir(os.path.join(d, 'train', 'images'))]
assert hits, 'tiles dataset not attached — add the prepare notebook output as input'
DATA_DIR = hits[0]
print('DATA_DIR =', DATA_DIR)


In [ ]:
# Train the models in parallel, one per GPU. A model failing (e.g. OOM) must
# never abort the notebook — finished checkpoints in runs/ are saved regardless.
import os, subprocess, sys, time

procs = []
for i, m in enumerate(MODELS):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i % 2))
    logf = open(f'{m}.log', 'w')
    p = subprocess.Popen(
        [sys.executable, '-m', 'roadx.train', '--model', m,
         '--data-dir', DATA_DIR, '--batch-size', str(BATCH_SIZE), '--out', 'runs'],
        stdout=logf, stderr=subprocess.STDOUT, env=env,
    )
    procs.append((m, p, logf))
    print(f'launched {m} on GPU {i % 2}')

def tail(fn):
    try:
        lines = [l.strip() for l in open(fn, errors='ignore') if 'epoch ' in l or 'done.' in l]
        return lines[-1] if lines else '(warming up...)'
    except FileNotFoundError:
        return '(no log yet)'

while any(p.poll() is None for _, p, _ in procs):
    time.sleep(300)
    for m, p, _ in procs:
        state = 'running' if p.poll() is None else f'exited {p.returncode}'
        print(f'[{m} | {state}] {tail(f"{m}.log")}', flush=True)

print('=' * 70)
for m, p, logf in procs:
    logf.close()
    print(f'{m}: exit code {p.returncode}')
    log = open(f'{m}.log', errors='ignore').read()
    print(log[-2500:])


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

for run in sorted(Path('runs').glob('*/log.csv')):
    df = pd.read_csv(run)
    plt.plot(df['epoch'], df['val_iou'], label=run.parent.name)
    print(run.parent.name, 'best val IoU:', df['val_iou'].max())
plt.xlabel('epoch'); plt.ylabel('val IoU'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Validation IoU'); plt.show()
